In [5]:
from gitsource import GithubRepositoryDataReader, chunk_documents

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)



files = reader.read()

print(vars(files[0]).keys())
files_dict_list = [{'filename': file.filename, 'content': file.content} for file in files]
print(len(files_dict_list))
chunks = chunk_documents(files_dict_list, size=2000, step=1000)
print(len(chunks))
print(chunks[0].keys())
print(files_dict_list[0])

dict_keys(['filename', 'content'])
72
295
dict_keys(['start', 'content', 'filename'])
{'filename': '01-agentic-rag/lessons/01-intro.md', 'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you

In [2]:
# Q1: How many lesson pages are in the dataset?
print(len(files_dict_list))
print(files_dict_list[0])

72
{'filename': '01-agentic-rag/lessons/01-intro.md', 'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is 

In [3]:
# Q2. Indexing and searching
# Index the documents with minsearch - make content a text field and filename a keyword field. Then search with this query:
# How does the agentic loop keep calling the model until it stops?
# What's the filename of the first result?

from repo_ingest import build_index
index = build_index(chunks)
query = "How does the agentic loop keep calling the model until it stops?"
result = index.search(query, num_results=5, boost_dict={'content': 3.0, 'filename': 0.5})
print(result[0]['filename'])

01-agentic-rag/lessons/14-agentic-loop.md


In [4]:
### Now we will build a RAG assistant on top of this data. Let's use the rag helper script we prepared during the lessons:

# wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
# RAGBase was written for the FAQ schema (section/question/answer), while our documents have filename and content.

# Two solutions are possible:

# Implement the RAG flow yourself
# Take RAGBase and change the parts related to the FAQ schema - search (to use our index) and build_context
# Build a RAG over the index from Q2 and answer the query:

# How does the agentic loop keep calling the model until it stops?

# Use gpt-5.4-mini. How many input (prompt) tokens did we send to the model for this request?
from repo_rag_helper import RAGBase
from openai import OpenAI

openai_client = OpenAI()

agent = RAGBase(
    index=index,
    llm_client=openai_client,
)
asnwer, input_tokens = agent.rag('How does the agentic loop keep calling the model until it stops?')
print(input_tokens)

ResponseUsage(input_tokens=2221, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=133, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=2354)
2221


In [6]:
# agentic rag
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback


In [12]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"content": 3.0, "filename": 0.5}
    )

agent_tools = Tools()
agent_tools.add_tool(search)
agent_tools.get_tools()

chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

INSTRUCTIONS = '''
You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.
'''

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=INSTRUCTIONS,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [13]:
result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)

-> Response received


-> Response received


Aspect,Plain RAG,Agentic loop
Control,application controls the flow,model controls next action
Retrieval,usually one search step,can search multiple times
Behavior,fixed pipeline,"iterative, dynamic"
Memory,often just query + retrieved docs,full conversation history
Flexibility,lower,higher
Cost/latency,usually lower,often higher


In [5]:
%load_ext autoreload
%autoreload 2